# Analytic nonlinear feature obfuscation with two MLP blocks
Consider a residual stream MLP architecture, like the one shown below:

![](model_architecture.svg)

The model is trying to learn `y = sat(x, -c, c)` (the saturation function, equivalently `y = max(-c, min(c, x))`). 
 `x` is a vector of dense features sampled uniformly from `[-3, 3]`, and `c` is a scalar sampled uniformly in `[1, 2]`. This is a fairly easy function to learn - it just takes `2*len(x)` neurons to learn exactly if you use ReLUs. But the harder question is, can a model do this without storing `c` in the residual stream, in a way that can be linearly probed?

This is an important question - although this is a toy model, its structure is reminiscent of how transformer models work, just on a much smaller scale. Normally, LLMs learn to store "features" as directions in activation space, such that you can train a linear probe to detect them. This is really useful! We can use this as a cheap monitor to detect things like whether the model is lying or whether the user's prompt is dangerous. You might think that if we can detect deception, we can just add the probe's score as a loss term in training, to align the model. However, training on mech-interp monitors like probes is considered [a forbidden technique](https://www.lesswrong.com/posts/mpmsK8KKysgSKDm2T/the-most-forbidden-technique), because the model could very well just learn to obfuscate its representations instead of learning what you wanted it to learn - and now you have a misaligned model *and* one less way to monitor it.

In this post I'll show that the model can at least theoretically represent `c` (itself meant to represent a feature we care about) in a non-linear fashion, in a way invisible to difference-of-means probes on every other layer, by means of an explicit construction (credit where credit's due: Claude came up with this encoding). Before we get started, though, I want to highlight some features of this architecture:
- The learned function is not linearly separable in `c` - so once the model has finished computing `y_i` for one `x_i` feature, it can just leave `y_i` in the residual stream. This is vaguely analogous to how probe accuracy seems to get a bit worse as you check the last layers of a transformer model, as the model starts shifting its representation from more abstract concepts to the exact text it wants to output. 
- Because it takes at least `2 * len(x)` neurons to implement `y`, as long as the width of the MLP blocks is less than `2 * len(x)`, we can be sure the model has represented `c` somehow in the first layer (or the model hasn't actually learned the task fully). More generally, the model needs to encounter at least `2 * len(x)` neurons before it can "forget" about `c`. In practical experiments, we can use this property to guarentee that the model is representing `c` at a particular early layer. 
- For simplicity, we'll also assume that the residual stream is arbitrarily wide, and that the embedding and unembedding matrices are rectangular identity matrices (i.e. ones on the main diagonal, zero elsewhere).

For notation: I'll use $W_i$ and $b_i$ to denote the MLP input weight and bias, and $W_o$ and $b_o$ for the output weight and bias.

In [ ]:

import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML

relu = lambda z: np.maximum(z, 0.0)
C_LO, C_HI = 1.0, 2.0
X_LO, X_HI = -3.0, 3.0


def animate_slider_html(param_values, build_frame, prefix, layout=None, height=450):
    """build_frame(p) -> (list_of_go_traces, dict_of_frame_layout_overrides_or_None)."""

    def frame_for(p):
        traces, layout_overrides = build_frame(p)
        return go.Frame(data=traces, name=f"{p:.3f}", layout=go.Layout(**(layout_overrides or {})))

    p0 = param_values[len(param_values) // 3]
    traces0, layout0 = build_frame(p0)
    fig = go.Figure(data=traces0, frames=[frame_for(p) for p in param_values])
    if layout:
        fig.update_layout(**layout)
    if layout0:
        fig.update_layout(**layout0)
    fig.update_layout(
        autosize=True,
        height=height,
        margin=dict(l=50, r=20, t=60, b=50),
        sliders=[{
            "currentvalue": {"prefix": prefix},
            "steps": [
                {
                    "method": "animate",
                    "label": f"{p:.2f}",
                    "args": [[f"{p:.3f}"], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}}],
                }
                for p in param_values
            ],
        }],
    )
    return HTML(fig.to_html(full_html=False, include_plotlyjs="cdn", config={"responsive": True}))


## Encoding: `v1(x1, c)` and `v2(x1, c)`
We'll encode `c` in the first MLP block, using two channels and borrowing an unrelated feature $x_1$:

```
v1(x1, c) = -2*ReLU(-x1 - c)   + 2*ReLU(x1 + c   - 3) - c + 1.5
v2(x1, c) = -4*ReLU(-x1 - c/2) + 4*ReLU(x1 + c/2 - 3) - c + 3.0
```
These have the property that $\int_{-3}^3 v_1 dx_1 = \int_{-3}^3 v_2 dx_1 = 0$. In other words, their mean value is always 0, regardless of $c$. This makes them invisible to difference-of-means probes. The next plot visualizes these functions. Note in particular the locations of the kinks:

- `v1`: `x1 = -c` and `x1 = 3-c`
- `v2`: `x1 = -c/2` and `x1 = 3-c/2`

Over the valid range of $1 \leq c \leq 2$, these four kink positions are *always* in the same left-to-right order: `-c < -c/2 < 3-c < 3-c/2`. This reliably carves `x1` into 5 bands.

As a bit of bookkeeping - we'll also erase `c` from the residual stream here, using an always-on neuron (e.g. add `-ReLU(c + 100) - 100)` to the channel contianing `c`).


In [ ]:
import sympy as sp

x_expr, c_expr = sp.symbols('x1 c', real=True)
relu_expr = lambda z: sp.Max(z, 0)

v1_expr = -2 * relu_expr(-x_expr - c_expr) + 2 * relu_expr(x_expr - 3 + c_expr) - c_expr + sp.Rational(3, 2)
v2_expr = -4 * relu_expr(-x_expr - c_expr/2) + 4 * relu_expr(x_expr + c_expr/2 - 3) - c_expr + 3

I1 = sp.simplify(sp.integrate(v1_expr, (x_expr, -3, 3)))
I2 = sp.simplify(sp.integrate(v2_expr, (x_expr, -3, 3)))

# As of writing - Sympy's inequality handling capabilities aren't great. Work around it by repeatedly substituting these known identities (since 1 <= c <= 2)
subs_facts = {
    sp.Max(-3, -c_expr): -c_expr,
    sp.Max(-3, 3 - c_expr): 3-c_expr,
    sp.Min(3, -c_expr): -c_expr,
    sp.Min(3, 3 - c_expr): 3-c_expr,
    sp.Max(-3, -c_expr/2): -c_expr/2,
    sp.Min(3, -c_expr/2): -c_expr/2,
    sp.Min(3, 3 - c_expr/ 2):3 - c_expr/ 2,
    sp.Max(-3, 3 - c_expr/ 2):3 - c_expr/ 2
}
def refine(expr):
    return sp.simplify(expr.subs(subs_facts))
print(refine(refine(I1))) # identically 0
print(refine(refine(I2)))


In [ ]:
def v1f(x, c):
    return -2 * relu(-x - c) + 2 * relu(x - 3 + c) - c + 1.5


def v2f(x, c):
    return -4 * relu(-x - c / 2) + 4 * relu(x + c / 2 - 3) - c + 3.0


x_grid = np.linspace(X_LO, X_HI, 121)
c_values = np.round(np.arange(C_LO, C_HI + 1e-9, 0.04), 2)


def step1_frame(c):
    v1, v2 = v1f(x_grid, c), v2f(x_grid, c)
    traces = [
        go.Scatter(x=x_grid, y=v1, name="v1(x1, c)", line=dict(color="#1f77b4", width=2)),
        go.Scatter(x=x_grid, y=v2, name="v2(x1, c)", line=dict(color="#ff7f0e", width=2)),
    ]
    shapes = [
        dict(type="line", x0=x, x1=x, y0=-6, y1=8, line=dict(color="#1f77b4", dash="dot", width=1))
        for x in (-c, 3 - c)
    ] + [
        dict(type="line", x0=x, x1=x, y0=-6, y1=8, line=dict(color="#ff7f0e", dash="dot", width=1))
        for x in (-c / 2, 3 - c / 2)
    ]
    return traces, {"shapes": shapes}


animate_slider_html(
    c_values,
    step1_frame,
    prefix="c = ",
    layout={
        "xaxis": {"title": "x1", "range": [X_LO, X_HI]},
        "yaxis": {"range": [-6, 8]},
        "title": "v1(x1) and v2(x1)  (dotted lines: kink positions)",
    },
)


## Decoding 

If you already knew which of the 5 bands `x1` was in, reading `c` back out
would be trivial, since each each segment of `v1` and `v2` is linear and can be inverted to recover `c`. In fact, you'd only need one of the `v` channels. For example, if you knew `x1 < -c`, then `v1` simplfies to `1.5-c` and you could recover `c = 1.5-v1`. 

Unfortunately, we can't predict ahead of time which band `x1` is going to be in, because we don't even know where the bands are (remember, we've erased `c`, which defines the kink locations). 
Fortunately (or unfortunately for AI safety people?), there's a workaround: it's possible to create affine functions $P(x_1, v_1, v_2)$ that cross the $P=0$ line only once, and always at an existing kink location ($\{-c, -c/2, 3-c, 3-c/2\}$). 

We'll get to how you can generate $P$ in a moment, but first - why is this useful? Well, in our decoding MLP block we can let $P$ be the ReLU input, and $R=\text{ReLU}(P)$ be the ReLU output. Notably, $R$ does not introduce any new kink locations.

Now, think about what it'd mean if we had a large number of $R_i$ available. Our goal is to reconstruct $c$ into the residual stream, so we have access to $n$ weight coefficients (one for each $R_i$), plus one more for the bias. Now, each $R_i$ is piecewise linear. For a given band (indexed by $b$), we can write $R_{i,b} = p_{i,b} x_1 + q_{i,b} c + r_{i,b}$. Then we need:
$$c = b + \sum^n_{i=1} w_i R_{i,b} = w_i p_{i,b} x_1 + w_i q_{i,b} c + w_i r_{i,b} + b = c$$
Considering the coefficients for $x$, $c$, and $1$, this gives us 3 linear equations:
$$ \sum^n_{i=1} w_i p_{i,b} = 0 $$
$$ \sum^n_{i=1} w_i q_{i,b} = 1 $$
$$ b + \sum^n_{i=1} w_i r_{i,b} = 0 $$
Since we have 5 bands, this we have 15 linear equations to satisfy, so we just need to come up with 15 $R_i$, that are sufficiently linearly independent. 

### Actually we don't need that many neurons
If you actually write out all these linear constraints, you'll find that the constraints aren't linearly independent. This is because we know $R$ is continuous at the boundaries between different bands. This means, for example, $\left.R_{i, b=0}\right|_{x=-c} = \left.R_{i, b=1}\right|_{x=-c}$. Looking at the coefficients for $c$ and $1$, we get two equations out of this equality - in other words, two redundant constraints. We can repeat this exercise for each of the four boundary conditions (kink locations), and find that there are 8 redundant constraints. This means we just need to find $15-8=7$ different $R_i$.

If you want to be fancier - you can observe that the space of piecewise linear functions we're considering has dimension 7 (6 from $x_1$, 1 from $c$), and therefore is spanned by a basis of 7 $R_i$.

We can bring the neuron count down even further - each of $v_1$, $v_2$, $x_1$, and $1$ are already accessible (the first three from the residual stream, the second from a bias term). We could use one always-on neuron for each of those to make them accessible to the output matrix/bias $W_o,b_o$, but we don't need to - we can fold their effects into the $W_i,b_i$ input matrix/bias of the next block. 

Overall, this means we only need *3* different $R_i$, and correspondingly 3 neurons in the decoding MLP block.

One more bookkeeping note - once block $n$ decodes a linear representation for $c$, block $n+1$ can use it and simultaneously erase it from the residual stream, so a probe at block $n+1$ can't detect $c$. Since the inputs $x_1, v_1, v_2$ are still present in the residual stream, block $n+2$ can repeat the process.

### Coming up with P
To find candidates for $P$, we express $P_i = a_0 + a_1 x + a_2 v_1 + a_3 v_2$ and pick one of the existing kink locations. Let's choose $x_1 = -c/2$ . We can substitute the expressions for $v_1$, $v_2$, and $x_1=-c/2$, and find that $P_i$ becomes a linear polynomial in $c$. We need $P_i$ to vanish identically, so that gives us 2 constraints (coefficients for $c$ and $1$) and 4 variables ($a_0, a_1, a_2, a_3$). Also, since scaling of $P_i$ doesn't give us independent $R_i$, we really only have one free parameter. We'll parameterize the space of $P_i$ with $\theta = \text{atan2}(a_3, a_2)$.

Then, we just have to validate that $P_i$ has only one zero-crossing, and that it is not fully positive or negative. (If it were, then $R_i$ wouldn't be linearly independent from $\{1,x_1, v_1, v_2\}$.) We do this numerically, and it turns out only $-c/2$ and $3-c/2$ are viable kink locations to vanish at.

The interactive tool below lets you explore the space of $P_i = a_0 + a_1 x_1 + a_2 v_1 + a_3 v_2$. $\theta$ sets $(a_2, a_3) = (\cos\theta, \sin\theta)$; $(a_0, a_1)$ are then solved for automatically. The curve is colored **green** when $P_i$ passes the strict one-sided test (checked across the full $c \in [1,2]$ range) and **red** otherwise.

In [ ]:
KINKS = ["-c", "-c/2", "3-c", "3-c/2"]
STATUS_GOOD, STATUS_CRITICAL = "#0ca30c", "#d03b3b"  # colorblind-safe status pair (icon+label always pairs with color)


def kink_x(kink, c):
    return {"-c": -c, "-c/2": -c / 2, "3-c": 3 - c, "3-c/2": 3 - c / 2}[kink]


def p_coeffs(kink, a2, a3):
    """(a0, a1) that make P_i = a0 + a1*x1 + a2*v1 + a3*v2 vanish identically along `kink`."""
    if kink == "-c":
        return -1.5 * a2 - 3 * a3, -a2 - 3 * a3
    if kink == "-c/2":
        return -1.5 * a2 - 3 * a3, -2 * a2 - 2 * a3
    if kink == "3-c":
        return 1.5 * a2, -a2 - a3
    if kink == "3-c/2":
        return 3 * a3 - 1.5 * a2, -2 * a3
    raise ValueError(kink)


def check_one_sided(kink, theta, x_grid=None, c_grid=None):
    """Same strict test as the atom search above: P must be strictly one sign on
    one side of the kink curve and non-positive on the other, for every c."""
    if x_grid is None:
        x_grid = np.linspace(X_LO, X_HI, 241)
    if c_grid is None:
        c_grid = np.linspace(C_LO, C_HI, 41)
    a2, a3 = np.cos(theta), np.sin(theta)
    a0, a1 = p_coeffs(kink, a2, a3)
    Xg, Cg = np.meshgrid(x_grid, c_grid, indexing="ij")
    P = a0 + a1 * Xg + a2 * v1f(Xg, Cg) + a3 * v2f(Xg, Cg)
    kpos = kink_x(kink, Cg)
    right, left = Xg > kpos + 1e-6, Xg < kpos - 1e-6
    pr, pl = (P[right] > 1e-9).mean(), (P[left] > 1e-9).mean()
    r_ok = pr > 1 - 1e-9 and pl < 1e-9
    l_ok = pl > 1 - 1e-9 and pr < 1e-9
    return r_ok or l_ok, ("R" if r_ok else ("L" if l_ok else None))


def build_p_i_tool(theta0_deg=0.0, c0=1.3, kink0="-c/2", div_id="pi-tool-plot"):
    x_plot = np.linspace(X_LO, X_HI, 301)
    a2_0, a3_0 = np.cos(np.radians(theta0_deg)), np.sin(np.radians(theta0_deg))
    a0_0, a1_0 = p_coeffs(kink0, a2_0, a3_0)
    v1_0, v2_0 = v1f(x_plot, c0), v2f(x_plot, c0)
    P0 = a0_0 + a1_0 * x_plot + a2_0 * v1_0 + a3_0 * v2_0
    valid0, _ = check_one_sided(kink0, np.radians(theta0_deg))
    color0 = STATUS_GOOD if valid0 else STATUS_CRITICAL

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x_plot, y=P0, mode="lines", name="P(x1)", line=dict(color=color0, width=2.5)))
    fig.add_trace(
        go.Scatter(x=x_plot, y=relu(P0), mode="lines", name="R = relu(P)", line=dict(color="#888888", width=1.5, dash="dot"))
    )
    fig.update_layout(
        shapes=[
            dict(
                type="line", x0=kink_x(k, c0), x1=kink_x(k, c0), y0=-6, y1=6,
                line=dict(
                    color="#111111" if k == kink0 else "#aaaaaa",
                    width=2 if k == kink0 else 1,
                    dash="solid" if k == kink0 else "dot",
                ),
            )
            for k in KINKS
        ],
        xaxis=dict(title="x1", range=[X_LO, X_HI]),
        yaxis=dict(title="P(x1), R(x1)", range=[-6, 6]),
        height=420,
        margin=dict(l=50, r=20, t=30, b=50),
        legend=dict(orientation="h", y=1.08),
    )
    plot_html = fig.to_html(include_plotlyjs="cdn", full_html=False, div_id=div_id, config={"responsive": True})

    kink_radios_html = "\n    ".join(
        f'<label><input type="radio" name="pi-tool-kink" value="{k}" {"checked" if k == kink0 else ""}> x1 = {k}</label>'
        for k in KINKS
    )

    controls_html = """
<div style="border:1px solid rgba(128,128,128,0.4); border-radius:8px; padding:16px 20px; margin:12px 0;">
  <div style="display:flex; flex-wrap:wrap; gap:28px; align-items:center; margin-bottom:10px;">
    <label style="display:flex; align-items:center; gap:10px;">
      <span>&theta; = <b id="pi-tool-theta-val">__THETA0__&deg;</b></span>
      <input id="pi-tool-theta" type="range" min="0" max="360" step="1" value="__THETA0__" style="width:220px;">
    </label>
    <label style="display:flex; align-items:center; gap:10px;">
      <span>c = <b id="pi-tool-c-val">__C0__</b></span>
      <input id="pi-tool-c" type="range" min="__C_LO__" max="__C_HI__" step="0.01" value="__C0__" style="width:220px;">
    </label>
  </div>
  <div style="display:flex; flex-wrap:wrap; gap:18px; align-items:center; margin-bottom:10px;">
    <span>kink curve to vanish along:</span>
    __KINK_RADIOS__
  </div>
  <div id="pi-tool-status" style="font-weight:600; margin-bottom:2px;"></div>
  <div id="pi-tool-coefs" style="font-family:monospace; font-size:0.85em; opacity:0.75; margin-bottom:10px;"></div>
  __PLOT_HTML__
</div>
<script>
(function () {
  const divId = "__DIV_ID__";
  const X_LO = __X_LO__, X_HI = __X_HI__, C_LO = __C_LO__, C_HI = __C_HI__;
  const GOOD = "__GOOD__", CRIT = "__CRIT__";
  const KINKS = ["-c", "-c/2", "3-c", "3-c/2"];

  function relu(z) { return Math.max(z, 0); }
  function v1(x, c) { return -2 * relu(-x - c) + 2 * relu(x - 3 + c) - c + 1.5; }
  function v2(x, c) { return -4 * relu(-x - c / 2) + 4 * relu(x + c / 2 - 3) - c + 3.0; }
  function kinkX(kink, c) {
    if (kink === "-c") return -c;
    if (kink === "-c/2") return -c / 2;
    if (kink === "3-c") return 3 - c;
    return 3 - c / 2;
  }
  function coeffs(kink, a2, a3) {
    if (kink === "-c") return [-1.5 * a2 - 3 * a3, -a2 - 3 * a3];
    if (kink === "-c/2") return [-1.5 * a2 - 3 * a3, -2 * a2 - 2 * a3];
    if (kink === "3-c") return [1.5 * a2, -a2 - a3];
    return [3 * a3 - 1.5 * a2, -2 * a3];
  }
  // Same strict one-sided test used to pick the atoms above: on a fine (x,c)
  // grid, P must be strictly positive on one side of the kink curve and
  // non-positive on the other, for every c in [C_LO, C_HI].
  function checkOneSided(kink, thetaRad) {
    const a2 = Math.cos(thetaRad), a3 = Math.sin(thetaRad);
    const [a0, a1] = coeffs(kink, a2, a3);
    const nx = 161, nc = 31;
    let totalR = 0, posR = 0, totalL = 0, posL = 0;
    for (let i = 0; i < nc; i++) {
      const c = C_LO + (C_HI - C_LO) * i / (nc - 1);
      const kpos = kinkX(kink, c);
      for (let j = 0; j < nx; j++) {
        const x = X_LO + (X_HI - X_LO) * j / (nx - 1);
        const P = a0 + a1 * x + a2 * v1(x, c) + a3 * v2(x, c);
        if (x > kpos + 1e-6) { totalR++; if (P > 1e-9) posR++; }
        else if (x < kpos - 1e-6) { totalL++; if (P > 1e-9) posL++; }
      }
    }
    const prFrac = totalR ? posR / totalR : 0, plFrac = totalL ? posL / totalL : 0;
    const rOk = prFrac > 1 - 1e-9 && plFrac < 1e-9;
    const lOk = plFrac > 1 - 1e-9 && prFrac < 1e-9;
    return { valid: rOk || lOk, side: rOk ? "R" : (lOk ? "L" : null) };
  }

  const thetaSlider = document.getElementById("pi-tool-theta");
  const cSlider = document.getElementById("pi-tool-c");
  const kinkRadios = document.getElementsByName("pi-tool-kink");
  const thetaVal = document.getElementById("pi-tool-theta-val");
  const cVal = document.getElementById("pi-tool-c-val");
  const status = document.getElementById("pi-tool-status");
  const coefs = document.getElementById("pi-tool-coefs");

  function currentKink() {
    for (const r of kinkRadios) if (r.checked) return r.value;
    return "-c/2";
  }

  function update() {
    const thetaDeg = parseFloat(thetaSlider.value);
    const c = parseFloat(cSlider.value);
    const kink = currentKink();
    thetaVal.textContent = thetaDeg.toFixed(0) + "°";
    cVal.textContent = c.toFixed(2);

    const thetaRad = thetaDeg * Math.PI / 180;
    const a2 = Math.cos(thetaRad), a3 = Math.sin(thetaRad);
    const [a0, a1] = coeffs(kink, a2, a3);

    const n = 301;
    const xs = new Array(n), Ps = new Array(n), Rs = new Array(n);
    for (let j = 0; j < n; j++) {
      const x = X_LO + (X_HI - X_LO) * j / (n - 1);
      const P = a0 + a1 * x + a2 * v1(x, c) + a3 * v2(x, c);
      xs[j] = x; Ps[j] = P; Rs[j] = relu(P);
    }

    const { valid, side } = checkOneSided(kink, thetaRad);
    const color = valid ? GOOD : CRIT;

    Plotly.restyle(divId, { x: [xs, xs], y: [Ps, Rs], "line.color": [color, "#888888"] }, [0, 1]);
    Plotly.relayout(divId, {
      shapes: KINKS.map(k => {
        const kp = kinkX(k, c), sel = (k === kink);
        return {
          type: "line", x0: kp, x1: kp, y0: -6, y1: 6,
          line: { color: sel ? "#111111" : "#aaaaaa", width: sel ? 2 : 1, dash: sel ? "solid" : "dot" },
        };
      }),
    });

    status.textContent = valid
      ? ("✓ one-sided for every c ∈ [1,2] — valid atom (" + side + "-side positive)")
      : "✗ not one-sided for some c ∈ [1,2] — crosses zero more than once, or never changes sign";
    status.style.color = color;
    coefs.textContent = "P = " + a0.toFixed(3) + " + (" + a1.toFixed(3) + ")·x1 + ("
      + a2.toFixed(3) + ")·v1 + (" + a3.toFixed(3) + ")·v2   [kink: x1 = " + kink + "]";
  }

  thetaSlider.addEventListener("input", update);
  cSlider.addEventListener("input", update);
  for (const r of kinkRadios) r.addEventListener("change", update);
  update();
})();
</script>
"""
    controls_html = (
        controls_html.replace("__THETA0__", f"{theta0_deg:.0f}")
        .replace("__C0__", f"{c0:.2f}")
        .replace("__C_LO__", f"{C_LO}")
        .replace("__C_HI__", f"{C_HI}")
        .replace("__X_LO__", f"{X_LO}")
        .replace("__X_HI__", f"{X_HI}")
        .replace("__KINK_RADIOS__", kink_radios_html)
        .replace("__GOOD__", STATUS_GOOD)
        .replace("__CRIT__", STATUS_CRITICAL)
        .replace("__DIV_ID__", div_id)
        .replace("__PLOT_HTML__", plot_html)
    )
    return HTML(controls_html)


build_p_i_tool()

## Summary
- **Block 0**: Encode $c$ nonlinearly into $v_1$ and $v_2$, and erase $c$ from the residual stream
- **Block `2k`** (an unprobed layer): compute the 3 `R_i = relu(P_i(x1, v1, v2))`, and sum them together in the appropriate ratios to fresh direction in the residual stream.
- **Block `2k+1`** (a probed layer): Compute and use `c = affine(x1, v1, v2, R_1..3)` using the input matrix/bias, and erase the previous block's evidence ($R_i$ output direction in the residual stream) using an always-on neuron.